In [1]:
import boto3
import sagemaker

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [2]:
#check role
role = sagemaker.get_execution_role()
print(role)

arn:aws:iam::169569810201:role/LabRole


In [3]:
#check your bucket
s3 = boto3.client("s3")
response = s3.list_buckets()
for bucket in response['Buckets']:
    print(bucket['Name'])

sagemaker-us-east-1-169569810201


In [4]:
#package up a trained model to deploy it to AWS SageMaker
#deployment platforms expect a single compressed archive (model.tar.gz)
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(
        "artifacts/best_model.pkl",
        arcname="best_model.pkl"
    )

In [ ]:
#Upload model to S3, because deployment will take the model from s3
s3 = boto3.client("s3")

bucket_name = "sagemaker-us-east-1-169569810201"   

s3.upload_file(
    "model.tar.gz",
    bucket_name,
    "credit-model/model.tar.gz"
)

In [12]:
#check model can be run or not
from inference import model_fn

model = model_fn("artifacts")

In [ ]:
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

BUCKET = "sagemaker-us-east-1-169569810201"
MODEL_S3_KEY = "credit-model/model.tar.gz"
ENDPOINT_NAME = "credit-score-endpoint"

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"  

def get_lab_role_arn():
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]

def main():
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role:      {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint:  {ENDPOINT_NAME}")

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference.py",
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print("\nDeploying endpoint (5-8 minutes)...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME
    )

    print("Testing endpoint with sample data...")
    sample = {
        "instances": [
            [18, "Entrepreneur", 62717.24, 5035.44, 6, 6, 29, 54, 22, 1.82, 7,
             "Bad", 2590.63, 31.9, 218, "Yes", 370.02, 42.6, 330.92, "High", "Large",
             0, 1, 1, 3, 2, 1, 1, 0, 0]
        ]
    }

    runtime = boto3.client("sagemaker-runtime", region_name=REGION)
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=str(sample).replace("'", '"'),
    )
    print("\nSmoke test response:")
    print(response["Body"].read().decode("utf-8"))


if __name__ == "__main__":
    main()

Role:      arn:aws:iam::169569810201:role/LabRole
Model URI: s3://sagemaker-us-east-1-169569810201/credit-model/model.tar.gz
Endpoint:  credit-score-endpoint

Deploying endpoint (5-8 minutes)...
-----!Testing endpoint with sample data...

Smoke test response:
{"probabilities": [[0.761031083598684, 0.23472677093004402, 0.004242145471271723]], "predictions": [0], "labels": ["Poor"]}


In [ ]:
#check log here: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#logsV2:log-groups

In [ ]:
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")
ENDPOINT_NAME = "credit-score-endpoint"

# 1. Delete the endpoint
print(f"Deleting endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deletion triggered.")
except Exception as e:
    print(f"No endpoint found to delete: {e}")

# 2. Delete the endpoint configuration
print(f"Deleting endpoint configuration: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Endpoint configuration deletion triggered.")
except Exception as e:
    print(f"No config found to delete: {e}")

print("\nCleanup complete!")

Deleting endpoint: credit-score-endpoint...
Endpoint deletion triggered.
Deleting endpoint configuration: credit-score-endpoint...
Endpoint configuration deletion triggered.

Cleanup complete!
